### Libaraies

In [1]:
import os
import psycopg2

from dotenv import load_dotenv

from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

from sqlalchemy import Column, Integer, Float, String, DateTime
from sqlalchemy.sql import func

from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

from sqlalchemy import inspect

from sqlalchemy import desc

import warnings
warnings.filterwarnings('ignore')

### Step 1: Connect to PostgreSQL + verify

In [3]:
load_dotenv()

DB_HOST     = os.getenv('DB_HOST')
DB_PORT     = os.getenv('DB_PORT')
DB_NAME     = os.getenv('DB_NAME')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

In [4]:
try:
    connection = psycopg2.connect(
        host     = DB_HOST,
        port     = DB_PORT,
        dbname   = DB_NAME,
        user     = DB_USER,
        password = DB_PASSWORD
    )
    print("PostgreSQL connected")
    print(f"   Host     : {DB_HOST}:{DB_PORT}")
    print(f"   Database : {DB_NAME}")
    connection.close()

except Exception as e:
    print(f"Connection failed: {e}")

PostgreSQL connected
   Host     : localhost:5432
   Database : loan_risk_db


### Step 2: Create SQLAlchemy Engine

In [5]:
DATABASE_URL = (f"postgresql://{DB_USER}:{DB_PASSWORD}"
                f"@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [6]:
engine  = create_engine(DATABASE_URL, echo=False)
Session = sessionmaker(bind=engine)
Base    = declarative_base()

In [7]:
with engine.connect() as connection:
    print("SQLAlchemy engine created")
    print(f"   Dialect  : {engine.dialect.name}")
    print(f"   Database : {DB_NAME}")
    print(f"   Base     : {Base}")

SQLAlchemy engine created
   Dialect  : postgresql
   Database : loan_risk_db
   Base     : <class 'sqlalchemy.orm.decl_api.Base'>


### Step 3: Define Applicant Table

In [125]:
class Applicant(Base):
    """
    Applicant tracking table.
    One row per unique person (identified by CNIC).
    Tracks their full history across multiple visits.
    Note: payment tracking requires a separate loan
          management system — out of scope here.
    """
    __tablename__ = 'applicants'

    id                = Column(Integer, primary_key=True, autoincrement=True)
    cnic              = Column(String(15), unique=True, nullable=False)
    first_seen        = Column(DateTime, default=func.now())
    last_seen         = Column(DateTime, default=func.now())

    # Visit tracking
    total_visits      = Column(Integer, default=0)
    total_approved    = Column(Integer, default=0)
    total_rejected    = Column(Integer, default=0)
    last_decision     = Column(String(10))

    # Last loan details
    last_loan_amnt    = Column(Float)
    last_interest_rate= Column(Float)

    # Relationship to predictions
    predictions = relationship('Prediction', back_populates='applicant',cascade='all, delete-orphan')

In [126]:
print("Applicant table defined")
print(f"   Table name : {Applicant.__tablename__}")
print(f"   Columns    :")
for col in Applicant.__table__.columns:
    print(f"      {col.name:30s} {str(col.type)}")

Applicant table defined
   Table name : applicants
   Columns    :
      id                             INTEGER
      cnic                           VARCHAR(15)
      first_seen                     DATETIME
      last_seen                      DATETIME
      total_visits                   INTEGER
      total_approved                 INTEGER
      total_rejected                 INTEGER
      last_decision                  VARCHAR(10)
      last_loan_amnt                 FLOAT
      last_interest_rate             FLOAT


### Step 4: Define Prediction Table

In [127]:
class Prediction(Base):
    """
    Main predictions table.
    One row per API call — stores model outputs.
    """
    __tablename__ = 'predictions'

    id                      = Column(Integer, primary_key=True, autoincrement=True)
    timestamp               = Column(DateTime, default=func.now())
    applicant_id            = Column(Integer, ForeignKey('applicants.id'), nullable=True)

    # Classification
    decision                = Column(String(10))    # APPROVED / REJECTED
    default_probability     = Column(Float)
    threshold               = Column(Float)

    # Regression
    interest_rate           = Column(Float)         # None for existing loan

    # Clustering
    cluster_id              = Column(Integer)
    cluster_label           = Column(String(50))
    cluster_prob_high_value = Column(Float)
    cluster_prob_standard   = Column(Float)

In [128]:
print("Prediction table defined")
print(f"   Table name : {Prediction.__tablename__}")
print(f"   Columns    :")
for col in Prediction.__table__.columns:
    print(f"      {col.name:30s} {str(col.type)}")

Prediction table defined
   Table name : predictions
   Columns    :
      id                             INTEGER
      timestamp                      DATETIME
      applicant_id                   INTEGER
      decision                       VARCHAR(10)
      default_probability            FLOAT
      threshold                      FLOAT
      interest_rate                  FLOAT
      cluster_id                     INTEGER
      cluster_label                  VARCHAR(50)
      cluster_prob_high_value        FLOAT
      cluster_prob_standard          FLOAT


### Step 5: Define Input Table

In [129]:
class Input(Base):
    """
    Raw input features table.
    One row per prediction — stores what the user submitted.
    Linked to Prediction via prediction_id.
    """
    __tablename__ = 'inputs'

    id            = Column(Integer, primary_key=True, autoincrement=True)
    prediction_id = Column(Integer, ForeignKey('predictions.id'))
    timestamp     = Column(DateTime, default=func.now())

    # Raw input features
    loan_amnt                 = Column(Float)
    loan_int_rate             = Column(Float)
    loan_grade                = Column(String(5))
    loan_percent_income       = Column(Float)
    loan_intent               = Column(String(50))
    person_income             = Column(Float)
    person_age                = Column(Integer)
    person_emp_length         = Column(Float)
    person_home_ownership     = Column(String(20))
    cb_person_default_on_file = Column(String(5))

    # Relationship back to Prediction
    prediction = relationship('Prediction', back_populates='inputs')

In [130]:
print("Input table defined")
print(f"   Table name : {Input.__tablename__}")
print(f"   Columns    :")
for col in Input.__table__.columns:
    print(f"      {col.name:30s} {str(col.type)}")

Input table defined
   Table name : inputs
   Columns    :
      id                             INTEGER
      prediction_id                  INTEGER
      timestamp                      DATETIME
      loan_amnt                      FLOAT
      loan_int_rate                  FLOAT
      loan_grade                     VARCHAR(5)
      loan_percent_income            FLOAT
      loan_intent                    VARCHAR(50)
      person_income                  FLOAT
      person_age                     INTEGER
      person_emp_length              FLOAT
      person_home_ownership          VARCHAR(20)
      cb_person_default_on_file      VARCHAR(5)


### Step 6: Define Explanation Table

In [131]:
class Explanation(Base):
    """
    SHAP explanations table.
    Multiple rows per prediction — one row per SHAP feature.
    Linked to Prediction via prediction_id.
    """
    __tablename__ = 'explanations'

    id            = Column(Integer, primary_key=True, autoincrement=True)
    prediction_id = Column(Integer, ForeignKey('predictions.id'))
    timestamp     = Column(DateTime, default=func.now())

    # Which model this explanation belongs to
    task          = Column(String(20))     # classification / regression

    # SHAP details
    feature_name  = Column(String(100))
    readable_name = Column(String(100))
    feature_value = Column(Float)
    shap_impact   = Column(Float)
    direction     = Column(String(50))     # increases / decreases risk

    # Relationship back to Prediction
    prediction = relationship('Prediction', back_populates='explanations')

In [132]:
print("Explanation table defined")
print(f"   Table name : {Explanation.__tablename__}")
print(f"   Columns    :")
for col in Explanation.__table__.columns:
    print(f"      {col.name:30s} {str(col.type)}")

Explanation table defined
   Table name : explanations
   Columns    :
      id                             INTEGER
      prediction_id                  INTEGER
      timestamp                      DATETIME
      task                           VARCHAR(20)
      feature_name                   VARCHAR(100)
      readable_name                  VARCHAR(100)
      feature_value                  FLOAT
      shap_impact                    FLOAT
      direction                      VARCHAR(50)


### Step 7: Define ModelMetadata Table

In [133]:
class ModelMetadata(Base):
    """
    Model metadata table.
    Tracks which models are in production.
    One row per model version.
    """
    __tablename__ = 'model_metadata'

    id            = Column(Integer, primary_key=True, autoincrement=True)
    created_at    = Column(DateTime, default=func.now())

    # Model details
    task          = Column(String(20))     # classification / regression / clustering
    model_name    = Column(String(100))    # e.g. cls_LightGBM
    model_version = Column(String(20))     # e.g. v1.0
    threshold     = Column(Float, nullable=True)  # only for classification

In [134]:
print("ModelMetadata table defined")
print(f"   Table name : {ModelMetadata.__tablename__}")
print(f"   Columns    :")
for col in ModelMetadata.__table__.columns:
    print(f"      {col.name:30s} {str(col.type)}")

ModelMetadata table defined
   Table name : model_metadata
   Columns    :
      id                             INTEGER
      created_at                     DATETIME
      task                           VARCHAR(20)
      model_name                     VARCHAR(100)
      model_version                  VARCHAR(20)
      threshold                      FLOAT


### Step 8: Add Relationships

In [135]:
# Add relationships to Prediction table
Prediction.inputs       = relationship('Input',       back_populates='prediction',cascade='all, delete-orphan')
Prediction.explanations = relationship('Explanation', back_populates='prediction',cascade='all, delete-orphan')
Prediction.applicant    = relationship('Applicant',   back_populates='predictions')

###  Create All Tables

In [136]:
# Create all tables in PostgreSQL
Base.metadata.create_all(engine)

### Verify Tables in Database

In [137]:
# Verify tables were created
inspector = inspect(engine)
tables    = inspector.get_table_names()

print("All tables created in PostgreSQL")
print(f"\n   Tables found in '{DB_NAME}':")
for table in tables:
    cols = inspector.get_columns(table)
    print(f"\n{table} ({len(cols)} columns)")
    for col in cols:
        print(f"      {col['name']:30s} {str(col['type'])}")

All tables created in PostgreSQL

   Tables found in 'loan_risk_db':

model_metadata (6 columns)
      id                             INTEGER
      created_at                     TIMESTAMP
      task                           VARCHAR(20)
      model_name                     VARCHAR(100)
      model_version                  VARCHAR(20)
      threshold                      DOUBLE PRECISION

applicants (10 columns)
      id                             INTEGER
      cnic                           VARCHAR(15)
      first_seen                     TIMESTAMP
      last_seen                      TIMESTAMP
      total_visits                   INTEGER
      total_approved                 INTEGER
      total_rejected                 INTEGER
      last_decision                  VARCHAR(10)
      last_loan_amnt                 DOUBLE PRECISION
      last_interest_rate             DOUBLE PRECISION

predictions (11 columns)
      id                             INTEGER
      timestamp                  

### Step 9: CRUD Save Functions

In [138]:
def save_prediction(session, prediction_data):
    """
    Save model outputs to predictions table.
    
    prediction_data keys:
        classification — decision, probability, threshold
        regression     — predicted_interest_rate (None for existing loan)
        clustering     — cluster_id, cluster_label, probabilities
    """
    cls = prediction_data['classification']
    reg = prediction_data.get('regression', {})
    clu = prediction_data['clustering']

    prediction = Prediction(
        decision                = cls['decision'],
        default_probability     = cls['default_probability'],
        threshold               = cls['threshold'],
        interest_rate           = reg.get('predicted_interest_rate'),
        cluster_id              = clu['cluster_id'],
        cluster_label           = clu['cluster_label'],
        cluster_prob_high_value = clu['probabilities']['High Value Borrower'],
        cluster_prob_standard   = clu['probabilities']['Standard Borrower']
    )

    session.add(prediction)
    session.commit()
    session.refresh(prediction)
    return prediction

In [139]:
def save_input(session, prediction_id, input_data):
    """
    Save raw user inputs to inputs table.
    input_data — raw dict of what user submitted
    """
    input_record = Input(
        prediction_id             = prediction_id,
        loan_amnt                 = input_data.get('loan_amnt'),
        loan_int_rate             = input_data.get('loan_int_rate'),
        loan_grade                = input_data.get('loan_grade'),
        loan_percent_income       = input_data.get('loan_percent_income'),
        loan_intent               = input_data.get('loan_intent'),
        person_income             = input_data.get('person_income'),
        person_age                = input_data.get('person_age'),
        person_emp_length         = input_data.get('person_emp_length'),
        person_home_ownership     = input_data.get('person_home_ownership'),
        cb_person_default_on_file = input_data.get('cb_person_default_on_file')
    )

    session.add(input_record)
    session.commit()
    session.refresh(input_record)
    return input_record

In [140]:
def save_explanation(session, prediction_id, explanations):
    """
    Save SHAP explanations to explanations table.
    explanations — list of dicts, one per SHAP feature
    """
    saved = []
    for exp in explanations:
        explanation = Explanation(
            prediction_id = prediction_id,
            task          = exp.get('task'),
            feature_name  = exp.get('feature'),
            readable_name = exp.get('readable_name'),
            feature_value = exp.get('feature_value'),
            shap_impact   = exp.get('shap_impact'),
            direction     = exp.get('direction')
        )
        session.add(explanation)
        saved.append(explanation)

    session.commit()
    return saved

In [141]:
def get_or_create_applicant(session, cnic, prediction_data, input_data):
    """
    Get existing applicant by CNIC or create new one.
    Updates visit stats on every call.

    - First visit  → creates new applicant row
    - Return visit → updates existing row stats
    """
    applicant = session.query(Applicant).filter(Applicant.cnic == cnic).first()

    decision  = prediction_data['classification']['decision']
    loan_amnt = input_data.get('loan_amnt')
    reg       = prediction_data.get('regression', {})
    rate      = reg.get('predicted_interest_rate') if reg else None

    if applicant is None:
        # First visit — create new applicant
        applicant = Applicant(
            cnic               = cnic,
            total_visits       = 1,
            total_approved     = 1 if decision == 'APPROVED' else 0,
            total_rejected     = 1 if decision == 'REJECTED' else 0,
            last_decision      = decision,
            last_loan_amnt     = loan_amnt,
            last_interest_rate = rate
        )
        session.add(applicant)
        session.commit()
        session.refresh(applicant)
        print(f"   New applicant created — CNIC: {cnic}")

    else:
        # Return visit — update stats
        applicant.total_visits       += 1
        applicant.total_approved     += 1 if decision == 'APPROVED' else 0
        applicant.total_rejected     += 1 if decision == 'REJECTED' else 0
        applicant.last_decision       = decision
        applicant.last_loan_amnt      = loan_amnt
        applicant.last_interest_rate  = rate
        applicant.last_seen           = func.now()
        session.commit()
        session.refresh(applicant)
        print(f"   Returning applicant updated — CNIC: {cnic}")
    return applicant

In [142]:
print("Save functions defined")
print("   get_or_create_applicant() — first visit or return visit")
print("   save_prediction()         — saves model outputs")
print("   save_input()              — saves raw user inputs")
print("   save_explanation()        — saves SHAP explanations")

Save functions defined
   get_or_create_applicant() — first visit or return visit
   save_prediction()         — saves model outputs
   save_input()              — saves raw user inputs
   save_explanation()        — saves SHAP explanations


### Step 10: CRUD Query Functions

In [ ]:
def get_prediction_stats(session):
    """Get summary statistics of all predictions."""

    total    = session.query(Prediction).count()
    approved = session.query(Prediction).filter(
        Prediction.decision == 'APPROVED').count()
    rejected = session.query(Prediction).filter(
        Prediction.decision == 'REJECTED').count()
    avg_prob = session.query(
        func.avg(Prediction.default_probability)).scalar()
    avg_rate = session.query(
        func.avg(Prediction.interest_rate)).scalar()

    return {
        'total'                  : total,
        'approved'               : approved,
        'rejected'               : rejected,
        'avg_default_probability': round(float(avg_prob or 0), 4),
        'avg_interest_rate'      : round(float(avg_rate or 0), 4)
    }

In [ ]:
def get_applicant_by_cnic(session, cnic):
    """Get single applicant by CNIC."""
    return session.query(Applicant).filter(Applicant.cnic == cnic).first()

def get_applicant_history(session, cnic):
    """
    Get full prediction history for an applicant.
    Returns applicant info + all their predictions.
    """
    applicant = get_applicant_by_cnic(session, cnic)
    if not applicant:
        print(f"   No applicant found with CNIC: {cnic}")
        return None, []

    predictions = session.query(Prediction).filter(
        Prediction.applicant_id == applicant.id).order_by(
        desc(Prediction.timestamp)).all()

    return applicant, predictions

def delete_applicant_by_cnic(session, cnic):
    """
    Delete applicant and ALL related records by CNIC.
    Cascade deletes:
        applicant → predictions → inputs + explanations
    Args:
        session: SQLAlchemy session
        cnic   : applicant CNIC string
    Returns:
        bool: True if deleted, False if not found
    """
    applicant = get_applicant_by_cnic(session, cnic)
    if applicant:
        session.delete(applicant)
        session.commit()
        return True
    return False


In [ ]:
print("Query functions defined")
print("   get_prediction_stats()")
print("   get_applicant_by_cnic()")
print("   get_applicant_history()")
print("   delete_applicant_by_cnic()")

Query functions defined
   get_applicant_by_cnic()
   get_applicant_history()
   get_all_applicants()
   get_prediction_by_id()
   get_all_predictions()
   get_predictions_by_decision()
   get_input_by_prediction_id()
   get_explanations_by_prediction_id()
   get_prediction_stats()


### Step 11: Test Save + Query End to End

In [146]:
session = Session()

# ── Dummy data ──
prediction_data = {
    'classification': {
        'decision'           : 'REJECTED',
        'default_probability': 0.95,
        'threshold'          : 0.6
    },
    'regression': {
        'predicted_interest_rate': 18.78
    },
    'clustering': {
        'cluster_id'   : 0,
        'cluster_label': 'High Value Borrower',
        'probabilities': {
            'High Value Borrower': 0.9991,
            'Standard Borrower'  : 0.0009
        }
    }
}

input_data = {
    'loan_amnt'                : 15000.0,
    'loan_int_rate'            : 18.78,
    'loan_grade'               : 'F',
    'loan_percent_income'      : 0.43,
    'loan_intent'              : 'DEBTCONSOLIDATION',
    'person_income'            : 35000.0,
    'person_age'               : 28,
    'person_emp_length'        : 2.0,
    'person_home_ownership'    : 'RENT',
    'cb_person_default_on_file': 'N'
}

# ── Visit 1 — First time applying ───
print("VISIT 1 — First Application")

cnic      = '42101-1234567-1'
applicant = get_or_create_applicant(session, cnic, prediction_data, input_data)
pred      = save_prediction(session, prediction_data)

# Link prediction to applicant
pred.applicant_id = applicant.id
session.commit()

inp  = save_input(session, pred.id, input_data)

print(f"\n   Applicant ID     : {applicant.id}")
print(f"   CNIC             : {applicant.cnic}")
print(f"   Total visits     : {applicant.total_visits}")
print(f"   Total approved   : {applicant.total_approved}")
print(f"   Total rejected   : {applicant.total_rejected}")
print(f"   Last decision    : {applicant.last_decision}")
print(f"   Last loan amount : {applicant.last_loan_amnt}")

# ── Visit 2 — Same person applies again ───
print("VISIT 2 — Same Person Applies Again")

# This time approved with better profile
prediction_data_2 = {
    'classification': {
        'decision'           : 'APPROVED',
        'default_probability': 0.09,
        'threshold'          : 0.6
    },
    'regression': {
        'predicted_interest_rate': 10.99
    },
    'clustering': {
        'cluster_id'   : 1,
        'cluster_label': 'Standard Borrower',
        'probabilities': {
            'High Value Borrower': 0.0009,
            'Standard Borrower'  : 0.9991
        }
    }
}

input_data_2 = {**input_data,
                'loan_amnt'   : 5000.0,
                'loan_grade'  : 'A',
                'person_income': 65000.0}

applicant = get_or_create_applicant(session, cnic, prediction_data_2, input_data_2)
pred2     = save_prediction(session, prediction_data_2)

pred2.applicant_id = applicant.id
session.commit()

inp2 = save_input(session, pred2.id, input_data_2)

print(f"\n   Applicant ID     : {applicant.id}")
print(f"   CNIC             : {applicant.cnic}")
print(f"   Total visits     : {applicant.total_visits}")
print(f"   Total approved   : {applicant.total_approved}")
print(f"   Total rejected   : {applicant.total_rejected}")
print(f"   Last decision    : {applicant.last_decision}")
print(f"   Last loan amount : {applicant.last_loan_amnt}")

# ── Query full history ───
print("FULL HISTORY FOR THIS APPLICANT")

applicant, history = get_applicant_history(session, cnic)

print(f"\n   CNIC          : {applicant.cnic}")
print(f"   Total visits  : {applicant.total_visits}")
print(f"   Approved      : {applicant.total_approved}")
print(f"   Rejected      : {applicant.total_rejected}")
print(f"\n   Prediction History:")
print(f"   {'pred_id':<10} {'decision':<12} {'prob':<8} {'rate'}")
for p in history:
    print(f"   {p.id:<10} {p.decision:<12} "
          f"{p.default_probability:<8} {p.interest_rate}")

session.close()

VISIT 1 — First Application
   Returning applicant updated — CNIC: 42101-1234567-1

   Applicant ID     : 1
   CNIC             : 42101-1234567-1
   Total visits     : 3
   Total approved   : 1
   Total rejected   : 2
   Last decision    : REJECTED
   Last loan amount : 15000.0
VISIT 2 — Same Person Applies Again
   Returning applicant updated — CNIC: 42101-1234567-1

   Applicant ID     : 1
   CNIC             : 42101-1234567-1
   Total visits     : 4
   Total approved   : 2
   Total rejected   : 2
   Last decision    : APPROVED
   Last loan amount : 5000.0
FULL HISTORY FOR THIS APPLICANT

   CNIC          : 42101-1234567-1
   Total visits  : 4
   Approved      : 2
   Rejected      : 2

   Prediction History:
   pred_id    decision     prob     rate
   5          APPROVED     0.09     10.99
   4          REJECTED     0.95     18.78
   2          APPROVED     0.09     10.99
   1          REJECTED     0.95     18.78


In [147]:
def save_full_prediction(session, cnic, prediction_data, input_data, explanations):
    """
    Save complete prediction in a single function call.
    Handles applicant tracking + prediction + input + explanations.

    Args:
        session        : SQLAlchemy session
        cnic           : applicant CNIC string
        prediction_data: dict from pipeline (cls + reg + cluster)
        input_data     : dict of raw user inputs
        explanations   : list of SHAP explanation dicts

    Returns:
        dict: saved applicant, prediction, input, explanations
    """
    # Step 1 — get or create applicant
    applicant = get_or_create_applicant(session, cnic, prediction_data, input_data)

    # Step 2 — save prediction
    prediction = save_prediction(session, prediction_data)

    # Step 3 — link prediction to applicant
    prediction.applicant_id = applicant.id
    session.commit()

    # Step 4 — save input
    input_record = save_input(session, prediction.id, input_data)

    # Step 5 — save explanations
    exps = save_explanation(session, prediction.id, explanations)

    return {
        'applicant'   : applicant,
        'prediction'  : prediction,
        'input'       : input_record,
        'explanations': exps
    }


print("save_full_prediction() defined")
print("   Wraps all save steps into one call")
print("   Step 1 — get or create applicant")
print("   Step 2 — save prediction")
print("   Step 3 — link prediction to applicant")
print("   Step 4 — save input")
print("   Step 5 — save explanations")

save_full_prediction() defined
   Wraps all save steps into one call
   Step 1 — get or create applicant
   Step 2 — save prediction
   Step 3 — link prediction to applicant
   Step 4 — save input
   Step 5 — save explanations


In [148]:
# Test save_full_prediction()

session = Session()

prediction_data = {
    'classification': {
        'decision'           : 'APPROVED',
        'default_probability': 0.12,
        'threshold'          : 0.6
    },
    'regression': {
        'predicted_interest_rate': 11.50
    },
    'clustering': {
        'cluster_id'   : 1,
        'cluster_label': 'Standard Borrower',
        'probabilities': {
            'High Value Borrower': 0.001,
            'Standard Borrower'  : 0.999
        }
    }
}

input_data = {
    'loan_amnt'                : 8000.0,
    'loan_int_rate'            : 11.50,
    'loan_grade'               : 'B',
    'loan_percent_income'      : 0.15,
    'loan_intent'              : 'EDUCATION',
    'person_income'            : 55000.0,
    'person_age'               : 30,
    'person_emp_length'        : 5.0,
    'person_home_ownership'    : 'RENT',
    'cb_person_default_on_file': 'N'
}

explanations = [
    {
        'task'         : 'classification',
        'feature'      : 'loan_grade_x_loan_int_rate',
        'readable_name': 'Loan Grade x Interest Rate',
        'feature_value': 2.30,
        'shap_impact'  : -1.20,
        'direction'    : 'decreases default risk'
    },
    {
        'task'         : 'classification',
        'feature'      : 'person_income',
        'readable_name': 'Annual Income',
        'feature_value': 55000.0,
        'shap_impact'  : -0.85,
        'direction'    : 'decreases default risk'
    },
    {
        'task'         : 'regression',
        'feature'      : 'loan_grade',
        'readable_name': 'Loan Grade',
        'feature_value': 1.0,
        'shap_impact'  : 1.10,
        'direction'    : 'increases interest rate'
    }
]

# One single call
result = save_full_prediction(
    session         = session,
    cnic            = '42101-9999999-9',
    prediction_data = prediction_data,
    input_data      = input_data,
    explanations    = explanations
)

print(f"\nsave_full_prediction() test complete")
print(f"\n   Applicant:")
print(f"      CNIC          : {result['applicant'].cnic}")
print(f"      Total visits  : {result['applicant'].total_visits}")
print(f"      Last decision : {result['applicant'].last_decision}")
print(f"\n   Prediction:")
print(f"      ID            : {result['prediction'].id}")
print(f"      Decision      : {result['prediction'].decision}")
print(f"      Probability   : {result['prediction'].default_probability}")
print(f"      Interest Rate : {result['prediction'].interest_rate}")
print(f"\n   Input:")
print(f"      Loan Amount   : {result['input'].loan_amnt}")
print(f"      Loan Grade    : {result['input'].loan_grade}")
print(f"\n   Explanations saved : {len(result['explanations'])}")

session.close()

   Returning applicant updated — CNIC: 42101-9999999-9

save_full_prediction() test complete

   Applicant:
      CNIC          : 42101-9999999-9
      Total visits  : 2
      Last decision : APPROVED

   Prediction:
      ID            : 6
      Decision      : APPROVED
      Probability   : 0.12
      Interest Rate : 11.5

   Input:
      Loan Amount   : 8000.0
      Loan Grade    : B

   Explanations saved : 3


In [149]:
# Final DB Stats Check

session = Session()

# Overall stats
stats = get_prediction_stats(session)
print(" Overall Stats:")
print(f"   Total predictions : {stats['total']}")
print(f"   Approved          : {stats['approved']}")
print(f"   Rejected          : {stats['rejected']}")
print(f"   Avg default prob  : {stats['avg_default_probability']}")
print(f"   Avg interest rate : {stats['avg_interest_rate']}")

# All applicants
applicants = get_all_applicants(session)
print(f"\n All Applicants ({len(applicants)}):")
print(f"   {'cnic':<20} {'visits':<8} {'approved':<10} {'rejected':<10} {'last_decision'}")
print(f"   {'-' * 60}")
for a in applicants:
    print(f"   {a.cnic:<20} {a.total_visits:<8} "
          f"{a.total_approved:<10} {a.total_rejected:<10} "
          f"{a.last_decision}")

# All predictions
preds = get_all_predictions(session)
print(f"\n All Predictions ({len(preds)}):")
print(f"   {'id':<6} {'decision':<12} {'prob':<8} {'rate':<10} {'cluster'}")
print(f"   {'-' * 55}")
for p in preds:
    print(f"   {p.id:<6} {p.decision:<12} "
          f"{p.default_probability:<8} "
          f"{p.interest_rate:<10} {p.cluster_label}")

session.close()

 Overall Stats:
   Total predictions : 6
   Approved          : 4
   Rejected          : 2
   Avg default prob  : 0.3867
   Avg interest rate : 13.7567

 All Applicants (2):
   cnic                 visits   approved   rejected   last_decision
   ------------------------------------------------------------
   42101-9999999-9      2        2          0          APPROVED
   42101-1234567-1      4        2          2          APPROVED

 All Predictions (6):
   id     decision     prob     rate       cluster
   -------------------------------------------------------
   6      APPROVED     0.12     11.5       Standard Borrower
   5      APPROVED     0.09     10.99      Standard Borrower
   4      REJECTED     0.95     18.78      High Value Borrower
   3      APPROVED     0.12     11.5       Standard Borrower
   2      APPROVED     0.09     10.99      Standard Borrower
   1      REJECTED     0.95     18.78      High Value Borrower


### Step 12: Document Report

In [150]:
# see in docs/database_report.md